# 06 - Evaluation: Models Evaluation Comparison


In this notebook, we perform a comparative analysis of all trained models using:
- Metric distribution analysis and comparison (plots and summary statistics)
- Statistical significance testing (Corrected Resampled t-test)


## Import libraries and set the paths

In [ ]:
from __future__ import annotations

import IPython.display as ipd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from fraud_dynamic_ensemble.config import MODELS_DIR, FIGURES_DIR
from fraud_dynamic_ensemble.evaluation.visual_evaluation import plot_model_distribution, plot_model_barplot
from fraud_dynamic_ensemble.evaluation.statistical_test_evaluation import compute_pairwise_corrected_resampled_ttest

In [ ]:
EXPERIMENT_NAME = "CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5"

In [ ]:
models_results_path: Path = MODELS_DIR / EXPERIMENT_NAME
print(f"Loading results at path:\n\t{models_results_path}")

In [ ]:
FIGURES_MODELS_COMPARISON_DIR = FIGURES_DIR / "EV_models_comparison_evaluation"
FIGURES_MODELS_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR = FIGURES_MODELS_COMPARISON_DIR / EXPERIMENT_NAME
FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MULTIPLE_MODELS_EVALUATION_DIR = FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR / "multiple_models_evaluation"
FIGURES_MULTIPLE_MODELS_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading and Basic Overview

In [ ]:
files = list(models_results_path.glob("*/generalization_metrics_summary.csv"))
generalization_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    generalization_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", generalization_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

In [ ]:
generalization_df

## Fix the metrics and model to evaluate and compare

In [ ]:
metrics_to_analyze = [
    "balanced_accuracy",
    "mcc",
    "average_precision",
    "f1",
]
print(f"Selected metrics:\n\t{metrics_to_analyze}")

In [ ]:
unique_models = generalization_df["model"].unique()
print(f"Selected models:\n\t{unique_models}")

## Model Comparison (Distribution Analysis)

### Objective
To compare the performance distribution of all models for a specific metric on a targeted data split (e.g., Generalization or Resubstitution).

### Methodology
We generate a **distribution plot** and a **statistical report** for the selected metric and split:
* **Scope:** Parametrizable (Default: **Generalization**).
* **Visualization:**
    * **Boxplot:** Displays the median, Interquartile Range (IQR), and spread.
    * **Strip Plot:** Overlays raw data points (100 folds) to visualize density.
* **Statistics:** A summary table (Mean, Std, Min, Max) is displayed and saved to quantify the visual results.
* **Sorting:** Models are sorted **Alphabetically** to ensure consistent ordering across different plots.
* **Scale:** Fixed to $0.0 - 1.0$ (or customized based on metric range).

### Interpretation Guide
* **Box Height:** Higher is better for standard metrics (e.g., F1, MCC). Lower is better for error metrics.
* **Box Size (IQR):** A short box implies high stability (low variance). A tall box implies the model behavior changes drastically depending on the data fold.
* **Split Comparison:** By generating this plot for both 'resubstitution' and 'generalization', one can visually confirm overfitting (if the Resubstitution box is high/tight while the Generalization box is low/wide).

In [ ]:
# List to store individual metric reports
metric_reports = []

for metric in metrics_to_analyze:
    report = plot_model_distribution(df=generalization_df,
                                     metric_name=metric,
                                     save_path=FIGURES_MULTIPLE_MODELS_EVALUATION_DIR,
                                     split_name="generalization")

    if report is not None:
        metric_reports.append(report)

# --- Final Aggregation ---
if metric_reports:
    # Concatenate all reports along columns (axis=1)
    # Since all reports share the same Index (Model Name), they will align automatically
    final_summary_df: pd.DataFrame = pd.concat(metric_reports, axis=1)

    print(f"\n{'='*80}\nFINAL DISTRIBUTION SUMMARY\n{'='*80}")
    ipd.display(final_summary_df)

    final_csv_path = FIGURES_MULTIPLE_MODELS_EVALUATION_DIR / "distribution_boxplot_summary.csv"
    final_summary_df.to_csv(final_csv_path)
else:
    print("No reports were generated.")

## Model Comparison (Mean Performance & Stability)

### Objective
To summarize the central tendency (Mean) and reliability (Standard Deviation) of each model on the specified data split.

### Methodology
We generate a **Bar Chart with Error Bars** and a **Summary Table**:
* **Calculation:** We compute the **Mean** and **Standard Deviation** (SD) across all data partitions (pooled Iterations × Folds).
    * **Bar Height:** Represents the Mean Score.
    * **Error Bar:** Represents ±1 Standard Deviation.
* **Formatted Report:** Statistics are aggregated into a concise "Mean ± SD" format (e.g., $0.852 \pm 0.012$) to facilitate quick comparison of stability.
* **Sorting:** Models are sorted **Alphabetically** for consistent cross-referencing.
* **Scale:** Fixed to $0.0 - 1.0$ (or customized) to ensure comparability.

### Interpretation Guide
* **Higher Bar:** Better average performance.
* **Shorter Error Bar:** More stable/reliable model.
* **Overlapping Error Bars:** If the error bars of two models overlap significantly, their performance difference may not be statistically significant (requires the t-test heatmap for confirmation).

In [ ]:
# List to store individual metric reports
metric_reports = []

for metric in metrics_to_analyze:
    report = plot_model_barplot(df=generalization_df,
                                metric_name=metric,
                                save_path=FIGURES_MULTIPLE_MODELS_EVALUATION_DIR,
                                split_name="generalization")

    if report is not None:
        metric_reports.append(report)

# --- Final Aggregation ---
if metric_reports:
    # Concatenate all reports along columns (axis=1)
    # Since all reports share the same Index (Model Name), they will align automatically
    final_summary_df: pd.DataFrame = pd.concat(metric_reports, axis=1)

    print(f"\n{'='*80}\nFINAL PERFORMANCE SUMMARY (Mean ± Std)\n{'='*80}")
    ipd.display(final_summary_df)

    final_csv_path = FIGURES_MULTIPLE_MODELS_EVALUATION_DIR / "distribution_barplot_summary.csv"
    final_summary_df.to_csv(final_csv_path)
else:
    print("No reports were generated.")

## Model Comparison & Statistical Significance Testing

### Objective
To definitively determine which models perform better than others by moving beyond simple average scores. We aim to identify **statistically significant differences** between model pairs, ensuring that observed performance gains are real and not merely artifacts of random data splitting.

### Methodology: Corrected Resampled t-test
Standard statistical tests (like the independent t-test) assume that data samples are independent. However, in Cross-Validation, training sets overlap significantly (e.g., 90% overlap in 10-fold CV), violating this assumption and leading to underestimated variance and high False Positive rates (Type I errors).

To address this, we employ the **Corrected Resampled t-test** proposed by Nadeau and Bengio (2003). This test adjusts the variance to account for the correlation between training sets and the ratio of testing to training samples.

The corrected t-statistic is calculated as:

$$t = \frac{\mu_{diff}}{\sqrt{\sigma_{diff}^2 \cdot (\frac{1}{n} + \frac{n_{test}}{n_{train}})}}$$

Where:
* $\mu_{diff}$: Mean difference in scores between Model A and Model B.
* $\sigma_{diff}^2$: Variance of the differences.
* $n$: Number of evaluations (Iterations $\times$ Folds).
* $\frac{n_{test}}{n_{train}}$: Correction factor for the overlap between training sets.

### Visualization: Pairwise Significance Heatmap
We visualize the results using a **Significance Heatmap** for each metric. This matrix displays the outcome of the hypothesis test for every pair of models based on the p-value.

* **Rows & Columns:** Each cell represents the comparison between the model on the row and the model on the column.
* **Color Coding:**
    * **Blue Cells ($p < 0.05$):** Indicate a **statistically significant difference**. Darker blues represent higher confidence (lower p-values, e.g., $p < 0.001$).
        * *Note:* This color confirms the models are *different*, but you must check the mean scores (or the CSV `Mean_Diff`) to see which one is *better*.
    * **Orange/Red Cells (NS):** Indicate **No Significant Difference**. The models perform statistically similarly, meaning any difference in their average scores is likely due to chance.

### Interpretation Guide
1.  **Identify "Cliques":** Look for blocks of **Orange (NS)** cells. These represent groups of models that are statistically indistinguishable from each other.
2.  **Verify Improvements:** If a high-scoring model has a **Blue** cell when compared to a baseline model, the improvement is real and statistically significant.
3.  **Symmetry:** The chart is symmetric. A Blue cell at (Row A, Col B) means A and B are significantly different. Use the `df_comparisons` table to confirm if A > B or B > A.

In [ ]:
pairwise_frames = []

for metric in metrics_to_analyze:
    df_result = compute_pairwise_corrected_resampled_ttest(
        df=generalization_df,
        metric_name=metric, # Pass single string here
        n_train=0.9,
        n_test=0.1,
        save_path=FIGURES_MULTIPLE_MODELS_EVALUATION_DIR
    )
    pairwise_frames.append(df_result)

# --- Aggregate & Save Master CSV ---
if pairwise_frames:
    # Concatenate all metric-specific dataframes vertically (stacking rows)
    final_comparisons_df: pd.DataFrame = pd.concat(pairwise_frames, axis=0)

    print(f"\n{'='*80}\nFINAL CORRECTED RESAMPLED t-TEST SUMMARY\n{'='*80}")
    ipd.display(final_comparisons_df)

    # Define filename
    master_csv_path = FIGURES_MULTIPLE_MODELS_EVALUATION_DIR / "corrected_resampled_ttest_summary.csv"
    final_comparisons_df.to_csv(master_csv_path, index=False)

else:
    print("No pairwise comparisons were generated.")